# fase 1b do Churn Intelligence — scraping e NLP das reclamações da Claro

nesse notebook, vamo pra segunda etapa da fase 1 do projeto. enquanto a fase 1a trabalhou com o dataset estruturado do Kaggle, aqui a gente coleta e processa dados não estruturados reais, com as reclamações públicas da Claro no Reclame Aqui.

o objetivo aqui é transformar o texto bruto de clientes insatisfeitos em features de sentimento que vão alimentar o pipeline de RAG na fase 3.

antes de começar a desenvolver o código em si, entrei na página da Claro no Reclame Aqui, analisei o DevTools e filtrei por Fetch/XHR na aba Network. na lista das requisições que estavam sendo feitas, procurei pelas requisições que retornavam um JSON com os dados das reclamações pra ver a URL completa, os headers enviados e a resposta JSON com os dados estruturados das reclamações.

a URL encontrada foi: https://iosearch.reclameaqui.com.br/raichu-io-site-search-v1/query/companyComplains/5/10?company=7712

e a lista completa dos headers foi:
- Accept: application/json, text/plain, */*,
- Accept-Encoding: gzip, deflate, br, zstd,
- Accept-Language: en-US,en;q=0.9,pt;q=0.8,
- Origin: https://www.reclameaqui.com.br,
- Referer: https://www.reclameaqui.com.br/,
- Sec-Ch-Ua: 'Chromium;v=146, Not-A.Brand;v=24, Google Chrome;v=146',
- Sec-Ch-Ua-Mobile: ?0,
- Sec-Ch-Ua-Platform: 'Windows',
- Sec-Fetch-Dest: empty,
- Sec-Fetch-Mode: cors,
- Sec-Fetch-Site: same-site,
- User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36

com essas informações, a gente já consegue começar a scrapear as reclamações

In [ ]:
# vamo tentar usar request simples e ver se funciona

import requests
import pandas as pd
import time

# vamo usar o "User-Agent" aqi pro site não bloquear por saber que é um bot
headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept-Language": "en-US,en;q=0.9,pt;q=0.8",
    "Origin": "https://www.reclameaqui.com.br",
    "Referer": "https://www.reclameaqui.com.br/",
    "Sec-Ch-Ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
}

reclamacoes = []

for offset in range(0, 500, 5):  #-> pra iterar de 0 a 500 pulando de 5 em 5
    url = f"https://iosearch.reclameaqui.com.br/raichu-io-site-search-v1/query/companyComplains/5/{offset}?company=7712"

    # pra fazer a requisição HTTP pra API, passando os headers que imitam um navegador
    response = requests.get(url, headers=headers)

    if response.status_code == 200: #-> 200 é "sucesso"
        data = response.json()
        # pra navegar pelo JSON até chegar na lista de reclamações
        complains = data['complainResult']['complains']['data']
        reclamacoes.extend(complains)
        print(f"offset {offset} — {len(complains)} reclamações coletadas")
    else:
        print(f"offset {offset} — erro {response.status_code}")

    # vamo usar um sleep de 1s entre requisições pra não sobrecarregar o servidor e evitar bloqueio
    time.sleep(1)

df_reclamacoes = pd.DataFrame(reclamacoes)
print(df_reclamacoes.shape)
print(df_reclamacoes.columns.tolist())

offset 0 — erro 403
offset 5 — erro 403
offset 10 — erro 403
offset 15 — erro 403
offset 20 — erro 403
offset 25 — erro 403
offset 30 — erro 403
offset 35 — erro 403
offset 40 — erro 403
offset 45 — erro 403
offset 50 — erro 403
offset 55 — erro 403
offset 60 — erro 403
offset 65 — erro 403
offset 70 — erro 403
offset 75 — erro 403
offset 80 — erro 403
offset 85 — erro 403
offset 90 — erro 403
offset 95 — erro 403
offset 100 — erro 403
offset 105 — erro 403
offset 110 — erro 403
offset 115 — erro 403
offset 120 — erro 403
offset 125 — erro 403
offset 130 — erro 403
offset 135 — erro 403
offset 140 — erro 403
offset 145 — erro 403
offset 150 — erro 403
offset 155 — erro 403
offset 160 — erro 403
offset 165 — erro 403
offset 170 — erro 403
offset 175 — erro 403
offset 180 — erro 403
offset 185 — erro 403
offset 190 — erro 403
offset 195 — erro 403
offset 200 — erro 403
offset 205 — erro 403
offset 210 — erro 403
offset 215 — erro 403
offset 220 — erro 403
offset 225 — erro 403
offset 230

o request chama a API certa e recebe o JSON estruturado, mas o Reclame Aqui usa Cloudflare, que bloqueia a requisição.

Quando um navegador real acessa o site, o Cloudflare executa JavaScript invisível que verifica dezenas de sinais, como movimento do mouse, fingerprint do navegador, tempo de carregamento, etc. só depois disso ele emite o cookie cf_clearance. o requests é uma biblioteca HTTP pura, o que significa que ele não executa JavaScript, então nunca consegue gerar esse cookie por conta própria.

então, em vez  de imitar um navegador, vamo usar Playwright que resolve isso controlando um navegador Chromium real, executando o JavaScript de verificação do Cloudflare normalmente e gerando o cookie válido. depois que o cookie existe na sessão do navegador, a gente faz o fetch da API de dentro do contexto do navegador, então o Cloudflare não consegue distinguir de um usuário real.

o Colab roda em servidores do Google, cujos IPs são imediatamente reconhecidos e bloqueados pelo Cloudflare. além disso, o Playwright precisa instalar o browser Chromium, que encontra alguns probelmas aqui no Colab. então vamo rodar o scraper localmente e depois retornar com o csv pra seguir aqui.

In [18]:
# vamo carregar o csv gerado pelo scraping com as reclamações da claro no reclame aqui:

import pandas as pd

url = "https://raw.githubusercontent.com/meurii/churn-intelligence/refs/heads/main/data/reclamacoes_claro.csv"
df_reclamacoes = pd.read_csv(url)
print(df_reclamacoes.shape)
df_reclamacoes.head()

(254, 69)


,lastReplyOrigin,oldComplainId,additionalFields,moderationUserName,companyName,otherProblemType,solved,dealAgain,otherProductType,titleMasked,...,userName,url,inModeration,moderateRequested,deleted,complainMediaInfos,fantasyName,category,descriptionMasked,user
0,NaN,NaN,NaN,****,Claro,NaN,False,NaN,Serviço Técnico Residencial,Reclamação sobre Atendimento Técnico: Serviço ...,...,****,reclamacao-sobre-atendimento-tecnico-servico-m...,False,NaN,False,[],Claro,255.0,Assunto: Reclamação Atendimento Técnico<br/><b...,NaN
1,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,Cobrança indevida de multa por portabilidade a...,...,****,cobranca-indevida-de-multa-por-portabilidade-a...,False,NaN,False,[],Claro,67.0,Fiz uma portabilidade a uns 4 anos atrás para ...,NaN
2,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,Claro: Assédio telefônico com ligações excessi...,...,****,claro-assedio-telefonico-com-ligacoes-excessiv...,False,NaN,False,[],Claro,66.0,Gostaria de registrar minha indignação com a q...,NaN
3,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,Cobrança duplicada e não estornada pela Claro ...,...,****,cobranca-duplicada-e-nao-estornada-pela-claro-...,False,NaN,False,[],Claro,67.0,Em dezembro de 2025 paguei a fatura de novembr...,NaN
4,NaN,NaN,NaN,****,Claro,NaN,False,NaN,NaN,Internet Empresarial com Reparo Técnico Penden...,...,****,internet-empresarial-com-reparo-tecnico-penden...,False,NaN,False,[],Claro,69.0,Já solicitei reparo técnico para a internet du...,NaN


apesar da Claro ter mais de 400 mil reclamações na plataforma, a gente só conseguiu registrar 254 no processo de scrapping, chegando a um limite de offset em torno de 255. é bem provável que essa limitação seja intencional, já que o Reclame Aqui oferece acesso completo aos dados via planos pagos da sua API comercial. como a intenção aqui é fazer uma exploração das possibilidades do modelo, a gente usou o acesso público via API interna que, provavelmente, restringe o acesso justamente pra proteger esse modelo de negócio.

tentativas de aumentar o tamanho da página via parâmetro na URL resultaram
em bloqueio de IP pelo Cloudflare. o limite de offset em 255 tbm se mostrou
consistente em múltiplas execuções independentes, confirmando que a API
pública simplesmente não entrega dados além desse ponto pra requisições
não autenticadas.

as reclamações que a gente conseguiu scrapear são informações reais, com texto completo, data, categoria e status de resolução. então, vamo considerar que isso é suficiente pra alimentar o pipeline de NLP e o sistema RAG da fase 3.

o foco do projeto continua sendo:

coleta → sentimento → modelo → explicabilidade → recomendação

e não apenas o volume alto e bruto de dados.

In [19]:
# vamo analisar os tipos e nulos das colunas

df_reclamacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 69 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   lastReplyOrigin              0 non-null      float64
 1   oldComplainId                0 non-null      float64
 2   additionalFields             0 non-null      float64
 3   moderationUserName           254 non-null    object 
 4   companyName                  254 non-null    object 
 5   otherProblemType             27 non-null     object 
 6   solved                       254 non-null    bool   
 7   dealAgain                    0 non-null      float64
 8   otherProductType             60 non-null     object 
 9   titleMasked                  254 non-null    object 
 10  type                         0 non-null      float64
 11  evaluation                   0 non-null      float64
 12  contentPoliciesViolation     3 non-null      object 
 13  score               

In [20]:
# vamo eliminar todas as colunas que tão completamente vazias

df_reclamacoes = df_reclamacoes.dropna(axis=1, how='all')
print(df_reclamacoes.shape)
df_reclamacoes.info() #-> pra gente ver as que sobraram

(254, 47)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 47 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   moderationUserName           254 non-null    object 
 1   companyName                  254 non-null    object 
 2   otherProblemType             27 non-null     object 
 3   solved                       254 non-null    bool   
 4   otherProductType             60 non-null     object 
 5   titleMasked                  254 non-null    object 
 6   contentPoliciesViolation     3 non-null      object 
 7   userState                    254 non-null    object 
 8   userRequestedDelete          254 non-null    bool   
 9   modified                     254 non-null    object 
 10  id                           254 non-null    object 
 11  evaluated                    254 non-null    bool   
 12  presence                     186 non-null    object 
 13  deletionRe

In [21]:
# vamo confirmar pelo id se tem alguma linha duplicada

df_reclamacoes.duplicated('id').sum()

np.int64(12)

In [22]:
# vamo remover as 12 linhas duplicatas que a gente encontrou

df_reclamacoes = df_reclamacoes.drop_duplicates(subset='id')
print(df_reclamacoes.shape)  # a gente espera que aqui tenham 242 linhas

(242, 47)


a gente ainda tem 47 colunas, mas nem todas são relevantes pro modelo.
- userEmail, userName, ip, deletedIp e address contém informações pessoais identificáveis de usuários. então, por questão de privacidade, vamo remover essas colunas;
- moderationUserName, companyName, companyShortname, fantasyName, titleMasked, descriptionMasked, maskingStatus, inModeration, category, moderateReason, moderationReasonDescription, frozen, deleted, deletionReason, userRequestedDelete, productType, otherProblemType, otherProductType, compliment, modified, read, complainMediaInfos, companyIndexes, legacyId, url, complainOrigin, failedToValidatePolicies, contentPoliciesViolation, company, contentViolatesPolicies e policiesViolationScore são, basicamente, dados operacionais internos da plataforma que não dizem nada sobre a experiência do cliente, então, vamo remover essas colunas tbm;
- por último, id era essencial só pra gente confirmar a deduplicação. como já fizemos essa etapa, dá pra remover ela tbm.

sobraram as colunas title, description, created, status, solved, hasReply, problemType, userCity, userState, evaluated, presence, interactions. vamo avaliar essas mais a fundo pra decidir quais seguem e quais devem ser removidas, pra manter apenas o que alimenta o NLP, o RAG e o que pode virar feature pro modelo.

In [23]:
# vamo confirmar a variabilidade dessas colunas, pra garantir que elas podem ser úteis pro modelo

print(f"Taxa de resposta da Claro: {df_reclamacoes['solved'].mean():.1%}")
print(f"Taxa de resposta da Claro: {df_reclamacoes['hasReply'].mean():.1%}")
print(f"Taxa de resposta da Claro: {df_reclamacoes['evaluated'].mean():.1%}")

Taxa de resposta da Claro: 0.0%
Taxa de resposta da Claro: 0.0%
Taxa de resposta da Claro: 0.0%


essa análise confirmou o que motivou a escolha da Claro como empresa do estudo:
taxa de resposta de 0% e taxa de resolução de 0% nas 242 reclamações coletadas.
esses dados, embora não entrem no modelo por falta de variabilidade, reforçam o
argumento de que um sistema de churn intelligence faria diferença real
aqui precisamente porque a empresa não tem nenhum processo de resposta ativo.

In [24]:
# vamo avaliar tbm os valores dessas colunas

print(df_reclamacoes['presence'].value_counts())
print(df_reclamacoes['interactions'].value_counts())
print(df_reclamacoes['status'].value_counts())
print(df_reclamacoes['created'].value_counts())
print(df_reclamacoes['problemType'].value_counts())

presence
SERVICE_OFF    176
STORE_ON         1
SERVICE_ON       1
Name: count, dtype: int64
interactions
[]    242
Name: count, dtype: int64
status
PENDING    242
Name: count, dtype: int64
created
2026-04-06T09:15:57    2
2026-04-06T11:18:08    1
2026-04-06T11:17:26    1
2026-04-06T11:17:20    1
2026-04-06T11:20:20    1
                      ..
2026-04-05T16:35:51    1
2026-04-05T16:35:43    1
2026-04-05T16:31:19    1
2026-04-05T16:27:27    1
2026-04-05T16:13:13    1
Name: count, Length: 241, dtype: int64
problemType
0000000000000075                     79
0000000000000875                     13
0000000000000877                     11
0000000000000011                      9
0000000000000868                      8
0000000000000017                      8
0000000000001227                      8
0000000000000078                      5
0000000000000878                      5
0000000000000007                      4
0000000000000045                      4
0000000000000639                     

além do evaluated, presence, interactions e status tbm apresentaram valor único em 100% dos registros. como já comentei, essa falta de variabilidade não agrega nada ao modelo. então elas tbm podem ser removidas. por último, problemType retorna códigos numéricos que, provavelmente, têm uma tabela de mapeamento interna do Reclame Aqui que não é pública. Sem acesso a essa tabela, os códigos não vão ter significado utilizável aqui.

concluindo: restam apenas as colunas title, description, created, userCity e userState. essas 5 colunas concentram toda a informação necessária pra gente:
- description e title alimentam o NLP e o RAG;
- created permite análise temporal;
- e userCity e userState dão contexto geográfico.

In [25]:
# vamo atualizar o df só com as colunas que vão ser relevantes

colunas_relevantes = ['title', 'description', 'created', 'userCity', 'userState']

df_final = df_reclamacoes[colunas_relevantes].copy()
print(df_final.shape)
# e aproveitar pra confirmar se tem algum dado faltando
print(df_final.isnull().sum())

(242, 5)
title          0
description    0
created        0
userCity       0
userState      0
dtype: int64


In [26]:
# vamo precisar fazer uma limpeza de texto nas colunas que vamo usar pra NLP e RAG
# pra isso, vamo começar instalando a biblioteca necessária

!pip install unidecode

In [27]:
# depois criar a função de limpeza

import re
from unidecode import unidecode

def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""

    # pra remover tags HTML
    texto = re.sub(r'<[^>]+>', ' ', texto)

    # pra remover emojis mantendo acentos
    texto = re.sub(r'[^\w\s\.,!?àáâãäéêëíîïóôõöúûüçñ]', ' ', texto)

    # pra converter para minúsculas
    texto = texto.lower()

    # pra remover URLs
    texto = re.sub(r'http\S+|www\S+', '', texto)

    # pra remover números de telefone e CPF (padrões comuns em reclamações)
    texto = re.sub(r'\d{2,}[\.\-]?\d{4,}[\.\-]?\d{4}', '', texto)

    # pra remover pontuação excessiva mantendo . e , que ajudam o modelo
    texto = re.sub(r'[!?]{2,}', '.', texto)  #-> !!! vira .
    texto = re.sub(r'[^\w\s\.,]', ' ', texto)

    # pra remover espaços extras
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

In [28]:
# e, por ultimo, vamo aplicar nas duas colunas de texto

df_final['description_clean'] = df_final['description'].apply(limpar_texto)
df_final['title_clean'] = df_final['title'].apply(limpar_texto)

# compara original vs limpo em alguns exemplos
for i in range(3):
    print("ORIGINAL:", df_final['description'].iloc[i][:200])
    print("LIMPO:", df_final['description_clean'].iloc[i][:200])
    print("---")

ORIGINAL: Assunto: Reclamação Atendimento Técnico<br/><br/>Prezados,<br/><br/>Venho formalizar reclamação sobre dois atendimentos técnicos r
LIMPO: assunto reclamação atendimento técnico prezados, venho formalizar reclamação sobre dois atendimentos técnicos r
---
ORIGINAL: Fiz uma portabilidade a uns 4 anos atrás para a Tim, sendo que fiquei mais de 2 nos com a claro e estão me cobrando multa, isso es
LIMPO: fiz uma portabilidade a uns 4 anos atrás para a tim, sendo que fiquei mais de 2 nos com a claro e estão me cobrando multa, isso es
---
ORIGINAL: Gostaria de registrar minha indignação com a quantidade absurda de ligações que eu tenho recebido da Claro. Todos os dias eu sou i
LIMPO: gostaria de registrar minha indignação com a quantidade absurda de ligações que eu tenho recebido da claro. todos os dias eu sou i
---


agora que a gente limpou o texto completamente, vamo passar pra etapa da NLP.
o BERTimbau é o modelo canônico pra sentimento em textos em português por ser o mais robusto pra produção, mas ele tem dois problemas práticos: ele baixa algo próximo de 500 MB e é lento em CPU. aqui a gente usa o Colab gratuito, sem GPU, então rodar em 242 textos (que é o nosso caso aqui), pode demorar bastante, além da sessão poder cair no meio.

então, vamo usar o pysentimiento, que foi treinado especificamente em textos em português do Brasil, incluindo textos de redes sociais e avaliações (que é um perfil similar ao das reclamações do Reclame Aqui). ele é leve, rápido em CPU, fácil de instalar, e retorna polaridade (positivo, negativo, neutro) com score de confiança. vai rodar em segundos nos nossos 242 textos.



In [29]:
# vamo passar pra análise de sentimento nas reclamações
# pra isso, vamo começar instalando a biblioteca necessária

!pip install pysentimiento

In [30]:
# vamo ver a distribuição de NEG, NEU e POS nas 242 reclamações

from pysentimiento import create_analyzer
analyzer = create_analyzer(task="sentiment", lang="pt")

df_final['sentimento'] = df_final['description_clean'].apply(
    lambda x: analyzer.predict(x[:512]).output
)

print(df_final['sentimento'].value_counts())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

sentimento
NEU    163
NEG     79
Name: count, dtype: int64


a análise retornou 163 neutros, 79 negativos e nenhum positivo (o que já era esperado, já que são reclamações, então positivo seria improvável). mas a quantidade de neutros é um pouco surpreendente, já que a gente esperava reclamações majoritariamente negativas. o que pode estar acontecendo aqui é que muitas reclamações podem estar sendo escritas em tom formal e descritivo, narrando o problema sem linguagem emocional explícita, e por isso, confundindo o modelo, que por ter sido treinado em textos de redes sociais (onde o sentimento costuma ser explícito), tem dificuldade com esse estilo mais burocrático.

vamo avaliar alguns exemplos de resultados neutros, pra ver se eles fazem sentido ou se são falsos neutros que deveriam ser negativos.

In [31]:
# vamo ver exemplos de reclamações classificadas como neutro
neutros = df_final[df_final['sentimento'] == 'NEU']['description_clean']
for texto in neutros[:10]:
    print(texto[:300])
    print("---")

assunto reclamação atendimento técnico prezados, venho formalizar reclamação sobre dois atendimentos técnicos r
---
em dezembro de 2025 paguei a fatura de novembro 2025 de maneira duplicada. acabei me confundindo pois a claro, por algum motivo, p
---
já solicitei reparo técnico para a internet duas vezes, mas ninguém resolveu ainda. o protocolo do último atendimento é .
---
bom dia em primeiro de dezembro deste ano fiz um contrato com a clara na pessoa do senhor yan no valor de 8490 de dezembro até hoj
---
tentei fazer um plano mais em conta para minha empresa na claro e os atendentes não tinham nenhum plano que fosse mais em conta, e
---
estou recebendo cobranças de um pagamento que eu já fiz.
---
contratei 5 assinaturas de wi fi mesh em 2023 por emergência, o sistema parou de funcionar, devolvi os 5 aparelhos e pedi cancelam
---
contratei um linha claro fixo em 25 02 26 com o serviço de portabilidade de vivo para claro.depois de 22 chamados abertos após a c
---
texto para excluir dív

o que dá pra perceber com alguns desses exemplos é que, realmente, são neutros mais pela forma do que pelo conteúdo. mesmo com textos descritivos e formais ("venho formalizar reclamação", "bom dia", "contratei", "pedi um chip"), algumas situações são graves, como 22 chamados abertos, pagamento duplicado, aparelhos devolvidos sem resolução. as reclamações são escritas de forma tão burocrática que o modelo não detecta negatividade, já que ele não classifica pelo conteúdo semântico da situação.

então, aqui a gente encontra uma limitação real do pysentimiento pra esse tipo de texto específico. mas de toda forma, mesmo que a feature de sentimento acabe não discriminando bem individualmente e não entre diretamente no modelo como feature preditiva forte, a distribuição NEG/NEU como um todo ainda é informação útil pro contexto narrativo no pipeline de RAG.

o entregável dessa fase é o 'reclamacoes_com_sentimento.csv', com o texto limpo de 242 reclamações reais da Claro, e com a classificação de sentimento de cada reclamação.

na fase 3, esse dataset vai alimentar o sistema de RAG, buscando semanticamente nas reclamações os contextos mais relevantes para gerar a recomendação de ação quando o modelo identificar o motivo principal do churn via SHAP.

In [32]:
# vamo remover as colunas de description e title com os textos originais
# tbm vamo renomear pra que as limpas não precisem ficar com _clean

df_final = df_final.drop(columns=['description', 'title'])
df_final = df_final.rename(columns={
    'description_clean': 'description',
    'title_clean': 'title'
})

print(df_final.columns.tolist())
print(f"shape: {df_final.shape}")
df_final.head()

['created', 'userCity', 'userState', 'description', 'title', 'sentimento']
shape: (242, 6)


,created,userCity,userState,description,title,sentimento
0,2026-04-06T11:18:09,São Paulo,SP,assunto reclamação atendimento técnico prezado...,reclamação sobre atendimento técnico serviço m...,NEU
1,2026-04-06T11:18:08,Florianópolis,SC,fiz uma portabilidade a uns 4 anos atrás para ...,cobrança indevida de multa por portabilidade a...,NEG
2,2026-04-06T11:17:26,Nova Iguaçu,RJ,gostaria de registrar minha indignação com a q...,claro assédio telefônico com ligações excessiv...,NEG
3,2026-04-06T11:17:20,São Paulo,SP,em dezembro de 2025 paguei a fatura de novembr...,cobrança duplicada e não estornada pela claro ...,NEU
4,2026-04-06T11:20:20,Rio de Janeiro,RJ,já solicitei reparo técnico para a internet du...,internet empresarial com reparo técnico penden...,NEU


In [42]:
# por último, vamo salvar o df gerado aqui localmente

from google.colab import files
files.download("reclamacoes_com_sentimento.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>